# COMP 3610 Assignment 4: MLOps and Model Deployment

**Name:** Varsha Roopchand  
**ID:** 816039243  
**Course:** COMP 3610 Big Data Analytics

This Assignment reuses the Assignment 2 taxi feature engineering pipeline, tracks experiments with MLflow, serves the best regression model through FastAPI, and documents Docker-based deployment.

## Notebook Setup

This section installs the project dependencies, defines the working paths, and imports the helper utilities used throughout the assignment. The reusable project code lives beside this notebook so the training, API, tests, and Docker workflow all stay consistent.

In [ ]:
! pip install -q -r requirements.txt

In [1]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd

ROOT = Path.cwd()
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print('Working directory:', ROOT)
print('Project files:')
for path in sorted(ROOT.iterdir()):
    print('-', path.name)

Working directory: c:\Users\earth\OneDrive\Documents\New project\assignment_4
Project files:
- .dockerignore
- .gitignore
- .pytest_cache
- __pycache__
- app.py
- assignment4.ipynb
- COMP3610_Assignment4.pdf
- data
- docker-compose.yml
- Dockerfile
- mlruns
- model_utils.py
- models
- README.md
- requirements.txt
- screenshots
- submission.txt
- test_app.py
- train_and_log.py


## Part 1: Experiment Tracking with MLflow

### Task 1.1: MLflow Setup and Experiment Logging

The next cells configure a local MLflow tracking directory, recreate the Assignment 2 feature pipeline, train at least two regression models, and log their parameters, metrics, tags, and model artifacts. The best model will later be reused by the FastAPI service.

run `mlflow ui --port 5000` in powershell

In [2]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from model_utils import train_and_select_best_model

EXPERIMENT_NAME = 'taxi-tip-prediction'
# Set up a local MLflow tracking server and configure your notebook to log to it 
mlflow.set_tracking_uri("http://localhost:5000")
# Create an MLflow experiment named "taxi-tip-prediction"
mlflow.set_experiment(EXPERIMENT_NAME)
print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment:', EXPERIMENT_NAME)

C:\Users\earth\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000
Experiment: taxi-tip-prediction


In [4]:
# Re-train (or load and evaluate) at least 2 models from Assignment 2 
training_result = train_and_select_best_model(sample_size=200_000)

summary_rows = []
for result in training_result['all_results']:
    row = {'model_name': result['model_name']}
    row.update(result['metrics'])
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows).sort_values('rmse').reset_index(drop=True)
results_df

,model_name,mae,rmse,r2
0,random_forest_regressor,1.241540,2.287701,0.631333
1,linear_regression,1.284799,2.356470,0.608836


The Random Forest Regressor outperformed the Linear Regression model across all three regression metrics, making it the better model for this task. It achieved a lower MAE (1.2415 vs 1.2848) and a lower RMSE (2.2877 vs 2.3565), showing that its predictions were closer to the true tip amounts on average and that it handled larger errors slightly better. It also achieved a higher R² score (0.6313 vs 0.6088), meaning it explained more of the variance in the target variable. Overall, these results suggest that the relationship between the taxi trip features and tip amount is not purely linear, and that the Random Forest was better able to capture the more complex patterns in the data.

In [5]:
from mlflow.tracking import MlflowClient

REGISTERED_MODEL_NAME = 'taxi-tip-regressor'
client = MlflowClient()
best_run_id = None
best_model_uri = None
best_rmse = None

for result in training_result['all_results']:
    pipeline = result['pipeline']
    model_name = result['model_name']
    metrics = result['metrics']

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(pipeline.named_steps['model'].get_params())
        mlflow.log_metrics(metrics)
        mlflow.set_tags({
            'model_type': type(pipeline.named_steps['model']).__name__,
            'dataset_version': training_result['dataset_version'],
        })

        signature = infer_signature(
            training_result['sample_input'],
            pipeline.predict(training_result['sample_input']),
        )
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path='model',
            signature=signature,
        )

        if best_rmse is None or metrics['rmse'] < best_rmse:
            best_rmse = metrics['rmse']
            best_run_id = run.info.run_id
            best_model_uri = f"runs:/{run.info.run_id}/model"

print('Best run id:', best_run_id)
print('Best model URI:', best_model_uri)

C:\Users\earth\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/16 12:28:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 12:28:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exe

🏃 View run linear_regression at: http://localhost:5000/#/experiments/1/runs/a150e34bb5e440889a60b4df8309579e
🧪 View experiment at: http://localhost:5000/#/experiments/1


C:\Users\earth\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/16 12:30:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 12:30:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exe

🏃 View run random_forest_regressor at: http://localhost:5000/#/experiments/1/runs/cf95b20966a3476d860b32dba6d93973
🧪 View experiment at: http://localhost:5000/#/experiments/1
Best run id: cf95b20966a3476d860b32dba6d93973
Best model URI: runs:/cf95b20966a3476d860b32dba6d93973/model


### Screenshots 
The required screenshot will be included in an uploaded folder in the repository

Both regression models were logged successfully to MLflow, and the experiment tracking output confirms that the `random_forest_regressor` was selected as the best-performing run. The warning messages shown during logging are informational rather than fatal: they relate to schema inference, a deprecated MLflow argument, and the general security considerations of pickle-based model serialization, but they did not prevent the models or experiment metadata from being saved correctly.


### Task 1.2: Model Comparison and Registry

This section compares the logged runs, registers the best-performing model, stores a descriptive version note, and demonstrates loading the registered model for inference. The comparison is based on the regression metrics required by the assignment: MAE, RMSE, and R?.

In [6]:
# Use the MLflow UI or API to compare all logged runs side-by-side; include a screenshot showing the comparison view 
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
comparison_df = runs_df[[
    'run_id',
    'tags.mlflow.runName',
    'metrics.mae',
    'metrics.rmse',
    'metrics.r2',
    'tags.model_type',
    'tags.dataset_version',
]].sort_values('metrics.rmse')
comparison_df

,run_id,tags.mlflow.runName,metrics.mae,metrics.rmse,metrics.r2,tags.model_type,tags.dataset_version
0,cf95b20966a3476d860b32dba6d93973,random_forest_regressor,1.241540,2.287701,0.631333,RandomForestRegressor,yellow_tripdata_2024-01.parquet
2,395e894c93794b55bcb4d812f1c58d7a,random_forest_regressor,1.241540,2.287701,0.631333,RandomForestRegressor,yellow_tripdata_2024-01.parquet
4,492ee155fda54f71ac88f1f75317183c,random_forest_regressor,1.241540,2.287701,0.631333,RandomForestRegressor,yellow_tripdata_2024-01.parquet
1,a150e34bb5e440889a60b4df8309579e,linear_regression,1.284799,2.356470,0.608836,LinearRegression,yellow_tripdata_2024-01.parquet
3,9e1387713651432fb7e6fcddaf398c56,linear_regression,1.284799,2.356470,0.608836,LinearRegression,yellow_tripdata_2024-01.parquet
5,ed8f6726ae9e4e3292641a8e7ee4beb0,linear_regression,1.284799,2.356470,0.608836,LinearRegression,yellow_tripdata_2024-01.parquet


### Write 2-3 sentences explaining which model performed best and why, referencing the logged metrics 
The random_forest_regressor performed best because it achieved the strongest results across all three logged metrics. It had the lowest MAE (1.241540) and RMSE (2.287701), meaning its predictions were closer to the true tip amounts on average, and it also had the highest R² (0.631333), showing that it explained more of the variation in tip_amount than the linear_regression model.

In [7]:
# Register your best-performing model in the MLflow Model Registry with a descriptive model name (e.g., "taxi-tip-regressor") and a version description documenting its performance 
model_version = mlflow.register_model(best_model_uri, REGISTERED_MODEL_NAME)
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=model_version.version,
    description=(
        f"Best Assignment 4 regression model with RMSE={best_rmse:.4f}. "
        f"Selected from the logged Assignment 2 regression candidates."
    ),
)

print('Registered model:', REGISTERED_MODEL_NAME)
print('Registered version:', model_version.version)

Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/16 12:35:49 WARNING mlflow.tracking._model_registry.fluent: Run with id cf95b20966a3476d860b32dba6d93973 has no artifacts at artifact path 'model', registering model based on models:/m-37b234b21e4e451f926f839608f4de15 instead
2026/04/16 12:35:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 3
Created version '3' of model 'taxi-tip-regressor'.


Registered model: taxi-tip-regressor
Registered version: 3


In [8]:
# Demonstrate loading the model from the registry and making a sample prediction
from model_utils import save_model_bundle

save_model_bundle(
    pipeline=training_result['best_pipeline'],
    metrics=training_result['best_metrics'],
    model_name=REGISTERED_MODEL_NAME,
    model_version=str(model_version.version),
    extra_metadata={
        'registered_model_name': REGISTERED_MODEL_NAME,
        'mlflow_tracking_uri': mlflow.get_tracking_uri(),
    },
)

sample_prediction = training_result['best_pipeline'].predict(training_result['sample_input'].head(1))[0]
print('Sample prediction from the best model:', round(float(sample_prediction), 2))
print('Saved deployment artifacts:')
print('-', MODELS_DIR / 'taxi_tip_model.joblib')
print('-', MODELS_DIR / 'model_info.json')

Sample prediction from the best model: 2.86
Saved deployment artifacts:
- c:\Users\earth\OneDrive\Documents\New project\assignment_4\models\taxi_tip_model.joblib
- c:\Users\earth\OneDrive\Documents\New project\assignment_4\models\model_info.json


## Part 2: Model Serving with FastAPI

### Task 2.1: API Design and Implementation

The API is implemented in `app.py`. It loads the trained model once at application startup, validates each request using Pydantic, and returns a structured JSON response with the prediction, model version, and a UUID prediction identifier for traceability.

run `uvicorn app:app --reload --port 8000` in powershell

In [9]:
# Check for app.py
from app import app

print('FastAPI title:', app.title)
print('FastAPI version:', app.version)
print('OpenAPI docs URL:', app.docs_url)

FastAPI title: Taxi Tip Prediction API
FastAPI version: 1.0.0
OpenAPI docs URL: /docs


The FastAPI application imported successfully, confirming that the API definition in `app.py` is valid and available to the notebook. The output shows the application title, version, and documentation route (`/docs`), which corresponds to the full Swagger UI address `http://localhost:8000/docs` when the server is running on port 8000.

In [10]:
example_payload = {
  "pickup_hour": 10,
  "pickup_day_of_week": 3,
  "is_weekend": False,
  "trip_duration_minutes": 15.0,
  "trip_speed_mph": 13.0,
  "log_trip_distance": 1.2,
  "fare_per_mile": 3.5,
  "fare_per_minute": 0.8,
  "pickup_borough": "Manhattan",
  "dropoff_borough": "Brooklyn",
  "passenger_count": 2,
  "trip_distance": 3.8,
  "fare_amount": 13.4
}

print(json.dumps(example_payload, indent=2))

{
  "pickup_hour": 10,
  "pickup_day_of_week": 3,
  "is_weekend": false,
  "trip_duration_minutes": 15.0,
  "trip_speed_mph": 13.0,
  "log_trip_distance": 1.2,
  "fare_per_mile": 3.5,
  "fare_per_minute": 0.8,
  "pickup_borough": "Manhattan",
  "dropoff_borough": "Brooklyn",
  "passenger_count": 2,
  "trip_distance": 3.8,
  "fare_amount": 13.4
}


### Task 2.2: Enhanced API Features

`app.py` also includes a batch prediction endpoint, a health check endpoint, a model information endpoint, and a global exception handler. These additions make the service easier to monitor, test, and troubleshoot while keeping error responses structured and safe for clients.

In [11]:
api_overview = {
    'POST /predict': 'Single prediction with validated taxi trip features.',
    'POST /predict/batch': 'Batch prediction for up to 100 trip records.',
    'GET /health': 'Service status and model version.',
    'GET /model/info': 'Model metadata, feature names, and training metrics.',
}
pd.Series(api_overview, name='description')

POST /predict          Single prediction with validated taxi trip fea...
POST /predict/batch         Batch prediction for up to 100 trip records.
GET /health                            Service status and model version.
GET /model/info        Model metadata, feature names, and training me...
Name: description, dtype: object

This is an example for testing
```json
{
  "pickup_hour": 10,
  "pickup_day_of_week": 3,
  "is_weekend": false,
  "trip_duration_minutes": 15.0,
  "trip_speed_mph": 13.0,
  "log_trip_distance": 1.2,
  "fare_per_mile": 3.5,
  "fare_per_minute": 0.8,
  "pickup_borough": "Manhattan",
  "dropoff_borough": "Brooklyn",
  "passenger_count": 2,
  "trip_distance": 3.8,
  "fare_amount": 13.4
}
```

### Task 2.3: API Testing

The automated tests in `test_app.py` cover successful single and batch predictions, validation failures, the health endpoint, model information retrieval, and an edge case involving an invalid zero-distance trip. The test suite creates a temporary model artifact so the API can be tested without rerunning the full training workflow.

In [12]:
# Write at least 5 test cases
test_result = subprocess.run(['python', '-m', 'pytest', '-q'], capture_output=True, text=True)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)

.......                                                                  [100%]
7 passed in 198.04s (0:03:18)



Part 2 showed that the deployed FastAPI service is functioning correctly for both inference and monitoring tasks. The POST /predict endpoint returned a valid prediction with a unique prediction_id and model_version, while invalid input correctly triggered an HTTP 422 response with a clear validation message, confirming that the Pydantic schema is enforcing the required constraints.

The enhanced API features also worked as expected: POST /predict/batch successfully returned predictions for multiple records in one request, GET /health confirmed that the service was running and that the model loaded successfully at startup, and GET /model/info exposed useful metadata about the deployed model, including its features and training metrics. Together with the passing pytest suite, these results indicate that the API is reliable, properly validated, and suitable for deployment as a structured prediction service.

### Screenshots 
The required screenshot will be included in an uploaded folder in the repository

## Part 3: Containerization with Docker

### Task 3.1: Dockerfile and Image Build

The Docker setup packages the FastAPI service and model artifacts into a portable container image. The container copies only the files needed to run inference, installs the pinned dependencies from `requirements.txt`, exposes port 8000, and starts the app with Uvicorn.

In [13]:
# Write a Dockerfile for your FastAPI application
dockerfile_text = (ROOT / 'Dockerfile').read_text(encoding='utf-8')
print(dockerfile_text)

FROM python:3.11-slim

WORKDIR /app

# Install Python dependencies before copying the rest of the project to keep rebuilds fast.
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

# Only copy the files needed to run the API in the container.
COPY app.py .
COPY model_utils.py .
COPY models ./models

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



The Docker image for the FastAPI prediction service was built successfully from the provided `Dockerfile`. The resulting custom image, `taxi-tip-api:latest`, had a size of approximately 1.7 GB (ran docker images in powershell), which reflects the included Python runtime, dependencies, and model artifacts required to serve predictions inside the container.


### Task 3.2: Docker Compose and Deployment Demo

The Compose file defines an `api` service that builds from the local Dockerfile, maps port 8000, and passes the model artifact locations through environment variables. Run the following commands in a terminal and capture screenshots of the running container and successful prediction calls:

1. `docker compose up --build`
2. Send at least three prediction requests to `http://localhost:8000/predict`
3. `docker compose down`

In [14]:
# Create a docker-compose.yml that orchestrates your deployment:
compose_text = (ROOT / 'docker-compose.yml').read_text(encoding='utf-8')
print(compose_text)

services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: taxi-tip-api
    environment:
      MODEL_PATH: /app/models/taxi_tip_model.joblib
      MODEL_INFO_PATH: /app/models/model_info.json
    ports:
      - "8000:8000"
    restart: unless-stopped



The deployment was successfully orchestrated with Docker Compose by starting the API service using `docker compose up -d`, sending multiple prediction requests to the containerized FastAPI endpoint, and shutting the service down cleanly with `docker compose down`. The API responded successfully to all three external prediction requests, confirming that the containerized application was accessible over the mapped port and that the deployment workflow operated correctly from start to finish.

The custom Docker image for the project, `taxi-tip-api:latest`, was built successfully and had a size of approximately 1.7 GB. The API container was configured to run on port 8000, with Docker mapping container port `8000` to host port `8000` so that the service could be accessed externally at `http://localhost:8000`.

To run the project successfully, Docker Desktop must be installed and running, and the trained model artifacts must already exist in the `models/` directory, specifically `taxi_tip_model.joblib` and `model_info.json`. The deployment also depends on the environment variables `MODEL_PATH` and `MODEL_INFO_PATH`, which are set in `docker-compose.yml` so that the FastAPI service can locate the saved model and metadata when the container starts.

The powershell commands used were:
For task 3.1
1. docker version
2. docker pull python:3.11-slim
3. docker build -t taxi-tip-api .
4. docker run -d -p 8000:8000 --name taxi-tip-api-container taxi-tip-api
5. docker ps
6. curl.exe http://localhost:8000/health
7.docker stop taxi-tip-api-container
8. docker rm taxi-tip-api-container
For task 3.2
9. docker compose up -d
10. docker compose ps
11. $body = @{
  pickup_hour = 18
  pickup_day_of_week = 5
  is_weekend = $true
  trip_duration_minutes = 22.0
  trip_speed_mph = 14.5
  log_trip_distance = 1.6
  fare_per_mile = 4.0
  fare_per_minute = 0.95
  pickup_borough = "Queens"
  dropoff_borough = "Manhattan"
  passenger_count = 1
  trip_distance = 5.2
  fare_amount = 20.9
} | ConvertTo-Json
Invoke-RestMethod -Uri "http://localhost:8000/predict" -Method Post -ContentType "application/json" -Body $body
12. $body2 = @{
  pickup_hour = 10
  pickup_day_of_week = 3
  is_weekend = $false
  trip_duration_minutes = 15.0
  trip_speed_mph = 13.0
  log_trip_distance = 1.2
  fare_per_mile = 3.5
  fare_per_minute = 0.8
  pickup_borough = "Manhattan"
  dropoff_borough = "Brooklyn"
  passenger_count = 2
  trip_distance = 3.8
  fare_amount = 13.4
} | ConvertTo-Json
Invoke-RestMethod -Uri "http://localhost:8000/predict" -Method Post -ContentType "application/json" -Body $body2
13. $body3 = @{
  pickup_hour = 8
  pickup_day_of_week = 1
  is_weekend = $false
  trip_duration_minutes = 12.0
  trip_speed_mph = 11.5
  log_trip_distance = 1.0
  fare_per_mile = 3.1
  fare_per_minute = 0.75
  pickup_borough = "Brooklyn"
  dropoff_borough = "Manhattan"
  passenger_count = 1
  trip_distance = 2.7
  fare_amount = 8.9
} | ConvertTo-Json
Invoke-RestMethod -Uri "http://localhost:8000/predict" -Method Post -ContentType "application/json" -Body $body3
14. docker compose down
15. docker images

# AI ASSISTANCE DISCLOSURE
AI (Chat GPT) was used in the process of creating this assignment for help understanding the project requirements, debugging, understanding the results and with the structure of code. All AI generated code was understood before submission.